# 🛡️ DeepShield Audio — Notebook 04: Evaluation

This notebook demonstrates how to evaluate trained models and calculate metrics:
- Accuracy, Precision, Recall, F1-Score
- Equal Error Rate (EER) and Optimal Thresholding
- ROC Curve and AUC
- Confusion Matrix

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, confusion_matrix, classification_report
import seaborn as sns

from src.evaluator import calculate_eer
from src.threshold_optimizer import find_optimal_threshold

plt.rcParams['figure.facecolor'] = '#1E1E2E'
plt.rcParams['axes.facecolor'] = '#1E1E2E'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = 'white'
plt.rcParams['xtick.color'] = 'white'
plt.rcParams['ytick.color'] = 'white'

print('✅ Imports successful')

## 1. Simulate Model Predictions
Instead of running a full evaluation over the dataset (which requires the dataset and a trained model), we'll simulate output probabilities to demonstrate the metrics.

In [ ]:
np.random.seed(42)

n_samples = 1000
# 0 = spoof, 1 = bonafide
y_true = np.concatenate([np.zeros(800), np.ones(200)])

# Simulate probabilities (model predictions)
# For spoof (label 0), probs should be low. For bonafide (label 1), probs should be high.
y_prob_spoof = np.random.beta(a=2, b=8, size=800)
y_prob_bonafide = np.random.beta(a=7, b=3, size=200)
y_pred_probs = np.concatenate([y_prob_spoof, y_prob_bonafide])

print("Simulated ground truth and predictions")

## 2. ROC Curve and AUC
The Receiver Operating Characteristic (ROC) curve plots True Positive Rate vs False Positive Rate at different classification thresholds.

In [ ]:
fpr, tpr, thresholds = roc_curve(y_true, y_pred_probs)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='#4ECDC4', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='#E63946', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc="lower right", facecolor='#2A2A35')
plt.show()

## 3. Equal Error Rate (EER) and Optimal Threshold
EER is the point where False Acceptance Rate (FAR) equals False Rejection Rate (FRR). This is the standard metric for the ASVspoof challenge.

In [ ]:
eer, eer_threshold = calculate_eer(y_true, y_pred_probs)
print(f"Equal Error Rate (EER): {eer:.4f}")
print(f"Threshold at EER:       {eer_threshold:.4f}")

# Alternatively, find threshold maximizing F1 score
opt_threshold = find_optimal_threshold(y_true, y_pred_probs)
print(f"Threshold for max F1:   {opt_threshold:.4f}")

## 4. Confusion Matrix and Classification Report
Using the optimal threshold, let's see the discrete classification results.

In [ ]:
y_pred_binary = (y_pred_probs >= opt_threshold).astype(int)

cm = confusion_matrix(y_true, y_pred_binary)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Spoof', 'Bonafide'], 
            yticklabels=['Spoof', 'Bonafide'])
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

print("Classification Report:")
print(classification_report(y_true, y_pred_binary, target_names=['Spoof', 'Bonafide']))